# 01b — EDA plots (TRAIN partition only)
Every plot below uses the **train** partition (80 trips) and says so in its title.
student_test is not used here, and evaluator data is never loaded.
Figures are saved to `results/figures/`.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import folium
import config
from src.data.loader import list_trips, load_trip
from src.data.preprocess import clean_track

PARTITION = "train"
GAP = float(config.GAP_THRESHOLD_S)
FIG = ROOT / "results" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.6,
                     "axes.axisbelow": True, "axes.edgecolor": MUTED, "text.color": INK,
                     "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED})
print("partition:", PARTITION, "| gap threshold:", GAP, "s")

partition: train | gap threshold: 30.0 s


In [2]:
rows, all_dt = [], []
for tid in list_trips(PARTITION):
    trip = load_trip(PARTITION, tid)
    clean, rep = clean_track(trip["track"])
    all_dt.append(clean["dt"].to_numpy()[1:])
    rows.append({"trip_id": tid, "points": rep["clean_points"], "max_gap_s": rep["max_gap_s"],
                 "lon_c": clean["lon"].mean(), "lat_c": clean["lat"].mean(),
                 "n_nodes": len(trip["nodes"]), "n_arcs": len(trip["arcs"])})
eda = pd.DataFrame(rows)
dt = np.concatenate(all_dt)
print(len(eda), "trips,", len(dt), "time steps")
print(f"share of steps exactly 1 s: {np.mean(dt == 1):.1%}")
eda.describe().round(1)

80 trips, 208337 time steps
share of steps exactly 1 s: 96.5%


,points,max_gap_s,lon_c,lat_c,n_nodes,n_arcs
count,80.0,80.0,80.0,80.0,80.0,80.0
mean,2605.2,2706.4,15.7,44.4,24168.6,48461.3
std,3312.3,18017.4,48.0,18.4,32272.3,64738.8
min,361.0,1.0,-100.3,-34.6,299.0,649.0
25%,945.8,1.8,-1.3,41.4,3487.8,7061.8
50%,1529.0,11.0,12.7,50.1,9608.0,19526.5
75%,2792.8,73.8,37.4,53.9,28846.8,58003.2
max,19313.0,153064.0,137.2,61.2,155003.0,307392.0


## 1. GPS points per trip

In [3]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(eda["points"], bins=30, color=BLUE, edgecolor="white", linewidth=1)
ax.axvline(eda["points"].median(), color=INK, lw=1, ls="--")
ax.text(eda["points"].median(), ax.get_ylim()[1]*0.92, f"  median {eda['points'].median():,.0f}", color=INK)
ax.set(title="GPS points per trip (train partition, n=80)", xlabel="points per trip", ylabel="number of trips")
fig.tight_layout(); fig.savefig(FIG / "points_per_trip_train.png"); plt.show()

## 2. Time between consecutive GPS points (log scale)

In [4]:
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.logspace(0, np.log10(dt.max()) + 0.1, 40)
ax.hist(dt, bins=bins, color=BLUE, edgecolor="white", linewidth=0.8)
ax.set_xscale("log"); ax.set_yscale("log")
ax.axvline(GAP, color=ORANGE, lw=1.5, ls="--")
ax.text(GAP, ax.get_ylim()[1]*0.3, f"  gap threshold {GAP:.0f} s", color=INK)
ax.set(title="Time step dt between GPS points (train partition, log-log)", xlabel="dt (seconds, log)", ylabel="count (log)")
fig.tight_layout(); fig.savefig(FIG / "dt_hist_train.png"); plt.show()
print(f"steps > {GAP:.0f} s: {(dt > GAP).sum()} of {len(dt)}")

steps > 30 s: 133 of 208337


## 3. Largest gap per trip

In [5]:
s = eda.sort_values("max_gap_s", ascending=False).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(range(len(s)), s["max_gap_s"], color=BLUE, width=0.8)
ax.set_yscale("log"); ax.axhline(GAP, color=ORANGE, lw=1.5, ls="--")
ax.text(len(s)-1, GAP*1.3, f"{GAP:.0f} s threshold", color=INK, ha="right")
for i in range(3):
    ax.text(i, s.loc[i, "max_gap_s"]*1.2, s.loc[i, "trip_id"][-3:], ha="center", fontsize=8, color=INK)
ax.set(title="Largest time gap per trip, sorted (train partition)", xlabel="trips (sorted by largest gap)", ylabel="max gap (s, log)")
ax.set_xticks([])
fig.tight_layout(); fig.savefig(FIG / "max_gap_per_trip_train.png"); plt.show()
print(f"trips with a gap > {GAP:.0f} s: {(eda.max_gap_s > GAP).sum()} of {len(eda)}")

trips with a gap > 30 s: 28 of 80


## 4. Where the trips are (trip centres)
PNG: longitude/latitude scatter. Interactive world map: `results/figures/trip_centres_train.html`.

In [6]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(eda["lon_c"], eda["lat_c"], s=36, color=BLUE, edgecolor="white", linewidth=0.8, zorder=3)
ax.set(xlim=(-180, 180), ylim=(-90, 90), xlabel="longitude (deg)", ylabel="latitude (deg)",
       title="Trip centres (train partition): mean GPS position of each trip")
ax.set_aspect("equal")
fig.tight_layout(); fig.savefig(FIG / "trip_centres_train.png"); plt.show()

world = folium.Map(location=[45, 30], zoom_start=2, tiles="OpenStreetMap")
for r in eda.itertuples():
    folium.CircleMarker([r.lat_c, r.lon_c], radius=5, color=BLUE, fill=True, fill_opacity=0.8,
                        tooltip=f"train/{r.trip_id}: {r.points} pts").add_to(world)
world.save(str(FIG / "trip_centres_train.html"))
print("lon range", eda.lon_c.min().round(2), "to", eda.lon_c.max().round(2),
      "| lat range", eda.lat_c.min().round(2), "to", eda.lat_c.max().round(2))

lon range -100.35 to 137.22 | lat range -34.62 to 61.25


## 5. Road network size per trip

In [7]:
s = eda.sort_values("n_arcs").reset_index(drop=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, col, colour, name in [(axes[0], "n_nodes", BLUE, "nodes (junctions)"), (axes[1], "n_arcs", ORANGE, "arcs (road segments)")]:
    ax.bar(range(len(s)), s[col], color=colour, width=0.8)
    ax.set_yscale("log"); ax.set_xticks([])
    ax.set(title=f"{name} per trip (train)", xlabel="trips (sorted by arcs)", ylabel=f"{name} (log)")
fig.suptitle("Road network size per trip (train partition)")
fig.tight_layout(); fig.savefig(FIG / "network_size_train.png"); plt.show()
eda[["n_nodes", "n_arcs"]].describe().round(0)

,n_nodes,n_arcs
count,80.0,80.0
mean,24169.0,48461.0
std,32272.0,64739.0
min,299.0,649.0
25%,3488.0,7062.0
50%,9608.0,19526.0
75%,28847.0,58003.0
max,155003.0,307392.0


In [8]:
eda.to_csv(FIG / "eda_trip_stats_train.csv", index=False)
print(sorted(p.name for p in FIG.iterdir()))

['candidates_student_test_00000000_100.png', 'dt_hist_train.png', 'eda_trip_stats_train.csv', 'max_gap_per_trip_train.png', 'network_size_train.png', 'points_per_trip_train.png', 'trip_centres_train.html', 'trip_centres_train.png']
